In [31]:
import json
import os
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from urllib.parse import urlparse
import msal
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(override=True)  # reload .env on every run

True

In [32]:
load_dotenv(override=True)  # pick up latest .env values

# Env var names on the left; your app IDs as defaults on the right.
TENANT_ID = os.getenv("AZURE_TENANT_ID", "ab8255df-0b3c-46e9-8c0d-11bc4b132da1")
CLIENT_ID = os.getenv("AZURE_CLIENT_ID", "764908e8-ed83-496d-8a08-e06fc28b8505")
CLIENT_SECRET = os.getenv("AZURE_CLIENT_SECRET", "")

# Root communication site: RAG AI Agentic Deployment Site (Test)
SHAREPOINT_SITE_URL = os.getenv(
    "SHAREPOINT_SITE_URL",
    "https://amenhanna.sharepoint.com",
)

LIST_NAMES: list[str] = []
LIBRARY_NAMES: list[str] = []

# "auto" = try app-only first, fall back to device-code login if permissions missing
AUTH_MODE = os.getenv("GRAPH_AUTH_MODE", "auto")

# Prompt once if secret is still missing and client_credentials is required
if AUTH_MODE == "client_credentials" and not CLIENT_SECRET:
    CLIENT_SECRET = getpass("Azure client secret (paste from Portal → Certificates & secrets): ")

GRAPH_BASE = "https://graph.microsoft.com/v1.0"
SCOPES = ["https://graph.microsoft.com/.default"]
DELEGATED_SCOPES = ["Sites.Read.All", "Files.Read.All"]

PROJECT_DIR = Path.cwd()
JSON_OUTPUT_DIR = PROJECT_DIR / "data" / "sharepoint_json"
JSON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not TENANT_ID or not CLIENT_ID:
    raise ValueError(
        "TENANT_ID and CLIENT_ID must be set. "
        "Use AZURE_TENANT_ID / AZURE_CLIENT_ID env vars or edit the defaults above."
    )

if AUTH_MODE == "client_credentials" and not CLIENT_SECRET:
    raise ValueError(
        "No client secret provided. Either:\n"
        "  1. Create a .env file with AZURE_CLIENT_SECRET=your-secret\n"
        "  2. Set AUTH_MODE=auto or AUTH_MODE=device_code in .env"
    )

print(f"Tenant: {TENANT_ID}")
print(f"Client: {CLIENT_ID}")
print(f"Auth mode: {AUTH_MODE}")
print(f"Client secret: {'set' if CLIENT_SECRET else 'missing'}")
print(f"SharePoint site: {SHAREPOINT_SITE_URL}")
print(f"JSON output: {JSON_OUTPUT_DIR}")

Tenant: ab8255df-0b3c-46e9-8c0d-11bc4b132da1
Client: 764908e8-ed83-496d-8a08-e06fc28b8505
Auth mode: client_credentials
Client secret: set
SharePoint site: https://amenhanna.sharepoint.com
JSON output: c:\Users\natna\Documents\RAG_Project\RAG_Application-\data\sharepoint_json


## 2. Authenticate with Microsoft Graph

**Option A — App-only (no login prompt):** Azure Portal → [your app API permissions](https://portal.azure.com/#view/Microsoft_AAD_RegisteredApps/ApplicationMenuBlade/~/CallAnAPI/appId/764908e8-ed83-496d-8a08-e06fc28b8505)
- Add **Application** permissions: `Sites.Read.All`, `Files.Read.All`
- Click **Grant admin consent**

**Option B — User login (fallback):** If admin consent isn't available, the cell below auto-falls back to device-code login.
- Add **Delegated** permissions: `Sites.Read.All`, `Files.Read.All`
- Authentication → **Allow public client flows** → **Yes**

In [33]:
REQUIRED_GRAPH_PERMS = {"Sites.Read.All", "Files.Read.All"}


def decode_token_claims(token: str) -> dict:
    import base64

    payload = token.split(".")[1]
    payload += "=" * (-len(payload) % 4)
    return json.loads(base64.urlsafe_b64decode(payload))


def decode_token_roles(token: str) -> list[str]:
    return decode_token_claims(token).get("roles", [])


def decode_token_scopes(token: str) -> set[str]:
    scp = decode_token_claims(token).get("scp", "")
    return set(scp.split()) if scp else set()


def _acquire_client_credentials_token(authority: str) -> dict:
    if not CLIENT_SECRET:
        raise ValueError("CLIENT_SECRET required for client_credentials auth.")
    app = msal.ConfidentialClientApplication(
        CLIENT_ID, authority=authority, client_credential=CLIENT_SECRET
    )
    return app.acquire_token_for_client(scopes=SCOPES)


def _acquire_device_code_token(authority: str) -> dict:
    app = msal.PublicClientApplication(CLIENT_ID, authority=authority)
    accounts = app.get_accounts()
    result = None
    if accounts:
        result = app.acquire_token_silent(DELEGATED_SCOPES, account=accounts[0])
    if not result:
        flow = app.initiate_device_flow(scopes=DELEGATED_SCOPES)
        if "user_code" not in flow:
            raise RuntimeError(f"Device flow failed: {flow}")
        print(flow["message"])
        result = app.acquire_token_by_device_flow(flow)
    return result


def get_access_token() -> tuple[str, str]:
    if not TENANT_ID or not CLIENT_ID:
        raise ValueError("TENANT_ID and CLIENT_ID cannot be empty.")

    authority = f"https://login.microsoftonline.com/{TENANT_ID}"
    mode = AUTH_MODE

    if mode == "auto":
        if CLIENT_SECRET:
            result = _acquire_client_credentials_token(authority)
            if "access_token" in result:
                roles = set(decode_token_roles(result["access_token"]))
                if REQUIRED_GRAPH_PERMS.issubset(roles):
                    print(f"Auth: client_credentials | roles: {', '.join(sorted(roles))}")
                    return result["access_token"], "client_credentials"
                missing = REQUIRED_GRAPH_PERMS - roles
                print(
                    "App-only token missing permissions: "
                    + ", ".join(sorted(missing))
                    + "\nFalling back to device-code login..."
                )
            else:
                print("App-only auth failed. Falling back to device-code login...")
        mode = "device_code"

    if mode == "client_credentials":
        result = _acquire_client_credentials_token(authority)
    else:
        result = _acquire_device_code_token(authority)

    if "access_token" not in result:
        error = result.get("error_description", str(result))
        if "AADSTS7000218" in error:
            raise RuntimeError(
                f"{error}\n\n"
                "Enable device-code login: Azure Portal → Authentication → "
                "Allow public client flows → Yes"
            )
        raise RuntimeError(error)

    token = result["access_token"]

    if mode == "client_credentials":
        roles = set(decode_token_roles(token))
        missing = REQUIRED_GRAPH_PERMS - roles
        if missing:
            raise PermissionError(
                "Missing application permissions: "
                + ", ".join(sorted(missing))
                + ".\nGrant admin consent in Azure Portal, or set AUTH_MODE=auto in .env"
            )
        print(f"Auth: client_credentials | roles: {', '.join(sorted(roles))}")
    else:
        scopes = decode_token_scopes(token)
        print(f"Auth: device_code | scopes: {', '.join(sorted(scopes)) or 'none'}")

    return token, mode


ACCESS_TOKEN, ACTUAL_AUTH_MODE = get_access_token()
HEADERS = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
print("Authenticated with Microsoft Graph.")

PermissionError: Missing application permissions: Files.Read.All, Sites.Read.All.
Grant admin consent in Azure Portal, or set AUTH_MODE=auto in .env

## 3. Graph helpers

In [27]:
def graph_get(url: str, params: dict | None = None) -> dict:
    """GET from Graph, following @odata.nextLink pagination."""
    items: list = []
    while url:
        response = requests.get(url, headers=HEADERS, params=params, timeout=60)
        if response.status_code == 401:
            roles = decode_token_roles(ACCESS_TOKEN)
            raise PermissionError(
                f"Graph 401 Unauthorized for {url}\n"
                f"Token roles: {roles or 'none'}\n"
                "Grant Application permissions Sites.Read.All + Files.Read.All "
                "and click 'Grant admin consent' in Azure Portal."
            )
        response.raise_for_status()
        payload = response.json()
        items.extend(payload.get("value", []))
        url = payload.get("@odata.nextLink")
        params = None
    return {"value": items}


def parse_sharepoint_site_url(site_url: str) -> tuple[str, str]:
    """Return (hostname, server-relative path) for Graph site lookup."""
    parsed = urlparse(site_url)
    hostname = parsed.netloc
    path = parsed.path.rstrip("/")
    return hostname, path


def build_site_graph_ref(hostname: str, site_path: str) -> str:
    """Build Graph site reference. Root site must be hostname:/ not hostname:"""
    if not site_path:
        return f"{hostname}:/"
    return f"{hostname}:{site_path}"


def get_site_id(site_url: str) -> str:
    hostname, site_path = parse_sharepoint_site_url(site_url)
    site_ref = build_site_graph_ref(hostname, site_path)
    url = f"{GRAPH_BASE}/sites/{site_ref}"
    response = requests.get(url, headers=HEADERS, timeout=60)
    if response.status_code == 401:
        roles = decode_token_roles(ACCESS_TOKEN)
        raise PermissionError(
            f"Graph 401 Unauthorized for {url}\n"
            f"Token roles: {roles or 'none'}\n"
            "Fix in Azure Portal → App registration → API permissions:\n"
            "  • Type: Application (not Delegated)\n"
            "  • Sites.Read.All\n"
            "  • Files.Read.All\n"
            "  • Then: Grant admin consent"
        )
    response.raise_for_status()
    site = response.json()
    print(f"Site: {site.get('displayName')} ({site['id']})")
    return site["id"]


SITE_ID = get_site_id(SHAREPOINT_SITE_URL)

HTTPError: 401 Client Error: Unauthorized for url: https://graph.microsoft.com/v1.0/sites/amenhanna.sharepoint.com:

## 4. Export SharePoint lists to JSON

In [ ]:
def get_sharepoint_lists(site_id: str) -> list[dict]:
    url = f"{GRAPH_BASE}/sites/{site_id}/lists"
    return graph_get(url)["value"]


def get_list_items(site_id: str, list_id: str) -> list[dict]:
    url = f"{GRAPH_BASE}/sites/{site_id}/lists/{list_id}/items"
    params = {"expand": "fields", "$top": "200"}
    raw_items = graph_get(url, params=params)["value"]

    cleaned = []
    for item in raw_items:
        fields = item.get("fields", {})
        cleaned.append({
            "id": item.get("id"),
            "webUrl": item.get("webUrl"),
            "createdDateTime": item.get("createdDateTime"),
            "lastModifiedDateTime": item.get("lastModifiedDateTime"),
            "fields": fields,
        })
    return cleaned


def export_lists_to_json(site_id: str, list_names: list[str] | None = None) -> Path:
    all_lists = get_sharepoint_lists(site_id)
    if list_names:
        selected = [lst for lst in all_lists if lst["displayName"] in list_names]
    else:
        # Skip hidden/system lists
        selected = [lst for lst in all_lists if not lst.get("list", {}).get("hidden", False)]

    export = {
        "source": "sharepoint_list",
        "siteUrl": SHAREPOINT_SITE_URL,
        "exportedAt": datetime.now(timezone.utc).isoformat(),
        "lists": [],
    }

    for lst in selected:
        items = get_list_items(site_id, lst["id"])
        export["lists"].append({
            "listId": lst["id"],
            "displayName": lst["displayName"],
            "webUrl": lst.get("webUrl"),
            "itemCount": len(items),
            "items": items,
        })
        print(f"List '{lst['displayName']}': {len(items)} item(s)")

    output_path = JSON_OUTPUT_DIR / "sharepoint_lists.json"
    output_path.write_text(json.dumps(export, indent=2, default=str), encoding="utf-8")
    print(f"Saved: {output_path}")
    return output_path


lists_json_path = export_lists_to_json(SITE_ID, LIST_NAMES or None)

In [ ]:
# Preview list data as a flat pandas DataFrame
lists_data = json.loads(lists_json_path.read_text(encoding="utf-8"))

rows = []
for lst in lists_data["lists"]:
    for item in lst["items"]:
        row = {"list": lst["displayName"], **item["fields"]}
        rows.append(row)

lists_df = pd.DataFrame(rows)
lists_df.head()

## 5. Export document library to JSON

In [ ]:
TEXT_EXTENSIONS = {".txt", ".md", ".csv", ".json", ".html", ".htm"}


def get_document_libraries(site_id: str) -> list[dict]:
    url = f"{GRAPH_BASE}/sites/{site_id}/drives"
    return graph_get(url)["value"]


def list_drive_items_recursive(drive_id: str, item_id: str = "root") -> list[dict]:
    url = f"{GRAPH_BASE}/drives/{drive_id}/items/{item_id}/children"
    children = graph_get(url)["value"]

    all_items = []
    for child in children:
        all_items.append(child)
        if "folder" in child:
            all_items.extend(list_drive_items_recursive(drive_id, child["id"]))
    return all_items


def download_file_text(drive_id: str, item_id: str) -> str | None:
    url = f"{GRAPH_BASE}/drives/{drive_id}/items/{item_id}/content"
    response = requests.get(url, headers=HEADERS, timeout=120)
    if response.status_code != 200:
        return None
    return response.text


def export_libraries_to_json(
    site_id: str,
    library_names: list[str] | None = None,
    include_file_content: bool = True,
) -> Path:
    drives = get_document_libraries(site_id)
    if library_names:
        drives = [d for d in drives if d["name"] in library_names]

    export = {
        "source": "sharepoint_document_library",
        "siteUrl": SHAREPOINT_SITE_URL,
        "exportedAt": datetime.now(timezone.utc).isoformat(),
        "libraries": [],
    }

    for drive in drives:
        items = list_drive_items_recursive(drive["id"])
        files = []

        for item in items:
            if "folder" in item:
                continue

            name = item.get("name", "")
            ext = Path(name).suffix.lower()
            file_record = {
                "id": item.get("id"),
                "name": name,
                "webUrl": item.get("webUrl"),
                "size": item.get("size"),
                "createdDateTime": item.get("createdDateTime"),
                "lastModifiedDateTime": item.get("lastModifiedDateTime"),
                "mimeType": item.get("file", {}).get("mimeType"),
                "content": None,
            }

            if include_file_content and ext in TEXT_EXTENSIONS:
                file_record["content"] = download_file_text(drive["id"], item["id"])

            files.append(file_record)

        export["libraries"].append({
            "driveId": drive["id"],
            "name": drive["name"],
            "webUrl": drive.get("webUrl"),
            "fileCount": len(files),
            "files": files,
        })
        print(f"Library '{drive['name']}': {len(files)} file(s)")

    output_path = JSON_OUTPUT_DIR / "sharepoint_libraries.json"
    output_path.write_text(json.dumps(export, indent=2, default=str), encoding="utf-8")
    print(f"Saved: {output_path}")
    return output_path


libraries_json_path = export_libraries_to_json(
    SITE_ID,
    LIBRARY_NAMES or None,
    include_file_content=True,
)

In [ ]:
# Preview document library metadata
libraries_data = json.loads(libraries_json_path.read_text(encoding="utf-8"))

file_rows = []
for lib in libraries_data["libraries"]:
    for f in lib["files"]:
        file_rows.append({
            "library": lib["name"],
            "name": f["name"],
            "size": f["size"],
            "hasContent": f["content"] is not None,
            "webUrl": f["webUrl"],
        })

files_df = pd.DataFrame(file_rows)
files_df.head()

## 6. Merge into one JSON file for chunking

This combined file is the input for the pre-chunking RAG step.

In [ ]:
def list_items_to_documents(lists_payload: dict) -> list[dict]:
    documents = []
    for lst in lists_payload.get("lists", []):
        for item in lst.get("items", []):
            documents.append({
                "sourceType": "sharepoint_list",
                "sourceName": lst["displayName"],
                "id": item.get("id"),
                "webUrl": item.get("webUrl"),
                "text": json.dumps(item.get("fields", {}), default=str),
            })
    return documents


def library_files_to_documents(libraries_payload: dict) -> list[dict]:
    documents = []
    for lib in libraries_payload.get("libraries", []):
        for f in lib.get("files", []):
            text = f.get("content") or f"[Binary or unsupported file: {f.get('name')}]"
            documents.append({
                "sourceType": "sharepoint_document_library",
                "sourceName": lib["name"],
                "id": f.get("id"),
                "webUrl": f.get("webUrl"),
                "text": text,
            })
    return documents


combined = {
    "source": "sharepoint_combined",
    "siteUrl": SHAREPOINT_SITE_URL,
    "exportedAt": datetime.now(timezone.utc).isoformat(),
    "documents": (
        list_items_to_documents(lists_data)
        + library_files_to_documents(libraries_data)
    ),
}

combined_path = JSON_OUTPUT_DIR / "sharepoint_combined.json"
combined_path.write_text(json.dumps(combined, indent=2, default=str), encoding="utf-8")

print(f"Combined documents: {len(combined['documents'])}")
print(f"Saved: {combined_path}")

rag_df = pd.DataFrame(combined["documents"])
rag_df.head()

## 7. Pre-chunking (next step)

Run this after JSON export. Each document's `text` field will be split into chunks.

In [ ]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200


def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start : start + chunk_size])
        start += chunk_size - overlap
    return chunks


chunk_rows = []
for _, row in rag_df.iterrows():
    for i, chunk in enumerate(chunk_text(str(row["text"]), CHUNK_SIZE, CHUNK_OVERLAP)):
        chunk_rows.append({
            "sourceType": row["sourceType"],
            "sourceName": row["sourceName"],
            "webUrl": row["webUrl"],
            "chunk_id": i,
            "text": chunk,
        })

chunks_df = pd.DataFrame(chunk_rows)
chunks_df.head()